# 02 — GDP Recovery Dynamic Baseline

This notebook comes after:

`01_Raw_Files_Coverage_Diagnostics.ipynb`

Purpose:

Construct observed GDP recovery outcome variables using the standardized `gdp_growth` series.

This notebook creates:

- `Recovery_2007`
- `Recovery_2019`

Core logic:

For the 2007 shock window:

- check 2008 and 2009;
- if the first negative GDP growth year is 2008, baseline year = 2007;
- if the first negative GDP growth year is 2009, baseline year = 2008.

For the 2019 shock window:

- check 2020 and 2021;
- if the first negative GDP growth year is 2020, baseline year = 2019;
- if the first negative GDP growth year is 2021, baseline year = 2020.

Special values:

- no negative growth in shock window → recovery value = `0`;
- not recovered by available data → recovery value = `20`;
- no shock-window data → recovery value is missing and flagged.

Important:

- This notebook uses GDP growth only.
- It does not use structural POSet variables.
- It does not impute missing GDP growth.
- It does not choose the final POSet country sample.
- Recovery outcomes are for validation / interpretation later, not for constructing the POSet order.

In [1]:
# ------------------------------------------------------
# IMPORTS AND PATHS
# ------------------------------------------------------

from pathlib import Path
from datetime import datetime
import pandas as pd
import numpy as np
import re
import shutil
import warnings

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)
pd.set_option("display.float_format", "{:.4f}".format)

PROJECT_ROOT = Path.cwd()

INPUT_FILE = (
    PROJECT_ROOT
    / "Data"
    / "Processed"
    / "00_Comparable_Raw_Files"
    / "Combined_Long"
    / "all_comparable_sources_long.csv"
)

PROCESSED_DIR = PROJECT_ROOT / "Data" / "Processed" / "Recovery"
DIAGNOSTICS_DIR = PROJECT_ROOT / "Data" / "Diagnostics" / "02_GDP_Recovery_Dynamic_Baseline"
AUDIT_DIR = PROJECT_ROOT / "Data" / "Audit"

RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
RUN_TIMESTAMP = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

CLEAR_PREVIOUS_OUTPUTS = True

if CLEAR_PREVIOUS_OUTPUTS:
    for folder in [PROCESSED_DIR, DIAGNOSTICS_DIR]:
        if folder.exists():
            shutil.rmtree(folder)

for folder in [PROCESSED_DIR, DIAGNOSTICS_DIR, AUDIT_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print("Run ID:", RUN_ID)
print("Input file:", INPUT_FILE.resolve())
print("Processed folder:", PROCESSED_DIR.resolve())
print("Diagnostics folder:", DIAGNOSTICS_DIR.resolve())
print("Audit folder:", AUDIT_DIR.resolve())

Run ID: 20260622_030128
Input file: D:\Milano Bicocca\Course Materials\1st Year\02 - Data Science Lab - 2526-1-FDS02Q003\03 Projects\Index\Notebooks\Data\Processed\00_Comparable_Raw_Files\Combined_Long\all_comparable_sources_long.csv
Processed folder: D:\Milano Bicocca\Course Materials\1st Year\02 - Data Science Lab - 2526-1-FDS02Q003\03 Projects\Index\Notebooks\Data\Processed\Recovery
Diagnostics folder: D:\Milano Bicocca\Course Materials\1st Year\02 - Data Science Lab - 2526-1-FDS02Q003\03 Projects\Index\Notebooks\Data\Diagnostics\02_GDP_Recovery_Dynamic_Baseline
Audit folder: D:\Milano Bicocca\Course Materials\1st Year\02 - Data Science Lab - 2526-1-FDS02Q003\03 Projects\Index\Notebooks\Data\Audit


In [2]:
# ------------------------------------------------------
# CONFIGURATION
# ------------------------------------------------------

GDP_GROWTH_VARIABLE = "gdp_growth"

NOT_RECOVERED_VALUE = 20

# Final Recovery_2007 / Recovery_2019 value definition:
#
# If recovered:
#     recovery_year - first_negative_growth_year
#
# Example:
#     if first negative year = 2020 and index returns to baseline in 2021,
#     the recovery value is 1.
#
# For transparency, the notebook also exports:
#     years_from_baseline_to_recovery
#     years_after_first_negative_to_recovery

FINAL_RECOVERY_TIME_DEFINITION = "years_after_first_negative_growth_year"

SHOCK_CONFIGS = [
    {
        "shock_id": "2007",
        "recovery_column": "Recovery_2007",
        "shock_window_years": [2008, 2009],
        "not_recovered_value": NOT_RECOVERED_VALUE,
    },
    {
        "shock_id": "2019",
        "recovery_column": "Recovery_2019",
        "shock_window_years": [2020, 2021],
        "not_recovered_value": NOT_RECOVERED_VALUE,
    },
]

print("GDP growth variable:", GDP_GROWTH_VARIABLE)
print("Final recovery time definition:", FINAL_RECOVERY_TIME_DEFINITION)
pd.DataFrame(SHOCK_CONFIGS)

GDP growth variable: gdp_growth
Final recovery time definition: years_after_first_negative_growth_year


,shock_id,recovery_column,shock_window_years,not_recovered_value
0,2007,Recovery_2007,"[2008, 2009]",20
1,2019,Recovery_2019,"[2020, 2021]",20


In [3]:
# ------------------------------------------------------
# LOAD STANDARDIZED GDP GROWTH DATA
# ------------------------------------------------------

if not INPUT_FILE.exists():
    raise FileNotFoundError(f"Input file not found: {INPUT_FILE}")

source = pd.read_csv(INPUT_FILE)

required_columns = {"Country", "Year", "variable", "Value"}

missing_columns = required_columns - set(source.columns)

if missing_columns:
    raise ValueError(f"Input file missing required columns: {missing_columns}")

source["Country"] = source["Country"].astype(str).str.strip().str.upper()
source["Year"] = pd.to_numeric(source["Year"], errors="coerce")
source["Value"] = pd.to_numeric(source["Value"], errors="coerce")

source = source.dropna(subset=["Country", "Year", "variable", "Value"]).copy()
source["Year"] = source["Year"].astype(int)

gdp_growth = source[source["variable"] == GDP_GROWTH_VARIABLE].copy()

if gdp_growth.empty:
    raise ValueError(f"No rows found for variable: {GDP_GROWTH_VARIABLE}")

# Keep useful metadata if present
metadata_cols = [
    col for col in [
        "Country",
        "country_name",
        "Year",
        "Value",
        "source_file",
        "source_family",
        "concept",
        "expected_unit",
    ]
    if col in gdp_growth.columns
]

gdp_growth = gdp_growth[metadata_cols].copy()

if "country_name" not in gdp_growth.columns:
    gdp_growth["country_name"] = gdp_growth["Country"]

duplicate_check = (
    gdp_growth
    .groupby(["Country", "Year"])
    .agg(
        row_count=("Value", "size"),
        unique_values=("Value", "nunique"),
        min_value=("Value", "min"),
        max_value=("Value", "max"),
    )
    .reset_index()
    .query("row_count > 1")
)

duplicate_check.to_csv(
    DIAGNOSTICS_DIR / "gdp_growth_duplicate_country_year_check.csv",
    index=False,
)

if len(duplicate_check.query("unique_values > 1")) > 0:
    display(duplicate_check)
    raise ValueError("Conflicting duplicate GDP growth Country-Year rows found.")

if len(duplicate_check) > 0:
    print("Only exact duplicate GDP growth rows found. Dropping exact duplicates.")
    gdp_growth = gdp_growth.drop_duplicates(subset=["Country", "Year", "Value"]).copy()

gdp_growth = gdp_growth.sort_values(["Country", "Year"]).reset_index(drop=True)

gdp_growth.to_csv(
    DIAGNOSTICS_DIR / "gdp_growth_input_cleaned.csv",
    index=False,
)

print("GDP growth rows:", len(gdp_growth))
print("Countries:", gdp_growth["Country"].nunique())
print("Years:", gdp_growth["Year"].min(), "-", gdp_growth["Year"].max())

display(gdp_growth.head())

GDP growth rows: 1131
Countries: 44
Years: 2000 - 2025


,Country,country_name,Year,Value,source_file,source_family,concept,expected_unit
0,AUS,Australia,2000,2.0315,OECD_GDP_Growth_2000_2025.csv,OECD_DBnomics,Real GDP growth,annual_percent_change
1,AUS,Australia,2001,3.9498,OECD_GDP_Growth_2000_2025.csv,OECD_DBnomics,Real GDP growth,annual_percent_change
2,AUS,Australia,2002,3.0913,OECD_GDP_Growth_2000_2025.csv,OECD_DBnomics,Real GDP growth,annual_percent_change
3,AUS,Australia,2003,4.2189,OECD_GDP_Growth_2000_2025.csv,OECD_DBnomics,Real GDP growth,annual_percent_change
4,AUS,Australia,2004,3.1555,OECD_GDP_Growth_2000_2025.csv,OECD_DBnomics,Real GDP growth,annual_percent_change


In [4]:
# ------------------------------------------------------
# GDP GROWTH INPUT COVERAGE
# ------------------------------------------------------

gdp_growth_coverage_summary = (
    gdp_growth
    .groupby("Country")
    .agg(
        country_name=("country_name", "first"),
        years_available=("Year", "nunique"),
        min_year=("Year", "min"),
        max_year=("Year", "max"),
        min_growth=("Value", "min"),
        median_growth=("Value", "median"),
        max_growth=("Value", "max"),
    )
    .reset_index()
    .sort_values(["years_available", "Country"], ascending=[False, True])
)

gdp_growth_coverage_summary.to_csv(
    DIAGNOSTICS_DIR / "gdp_growth_country_coverage_summary.csv",
    index=False,
)

display(gdp_growth_coverage_summary.head(80))

,Country,country_name,years_available,min_year,max_year,min_growth,median_growth,max_growth
0,AUS,Australia,26,2000,2025,-0.1335,2.6278,4.2530
1,AUT,Austria,26,2000,2025,-6.3183,1.6197,5.3310
2,BEL,Belgium,26,2000,2025,-4.7930,1.6341,6.2545
4,CAN,Canada,26,2000,2025,-5.0382,2.1879,5.9505
5,CHE,Switzerland,26,2000,2025,-2.2553,1.7142,6.1762
6,CHL,Chile,26,2000,2025,-6.1397,3.2557,11.3374
8,COL,Colombia,26,2000,2025,-7.1859,3.2352,10.8012
10,CZE,Czechia,26,2000,2025,-5.3049,2.7725,6.6232
11,DEU,Germany,26,2000,2025,-5.5446,1.1489,4.1346
12,DNK,Denmark,26,2000,2025,-4.9745,1.4878,6.5003


In [5]:
# ------------------------------------------------------
# RECOVERY CONSTRUCTION FUNCTIONS
# ------------------------------------------------------

def build_country_growth_map(country_df):
    country_df = country_df.sort_values("Year").copy()
    return dict(zip(country_df["Year"].astype(int), country_df["Value"].astype(float)))


def determine_first_negative_year(growth_by_year, shock_window_years):
    available_shock_values = {
        year: growth_by_year.get(year, np.nan)
        for year in shock_window_years
        if year in growth_by_year
    }

    if len(available_shock_values) == 0:
        return None, available_shock_values, "no_shock_window_data"

    negative_years = [
        year
        for year in sorted(available_shock_values)
        if pd.notna(available_shock_values[year]) and available_shock_values[year] < 0
    ]

    if len(negative_years) == 0:
        return None, available_shock_values, "no_negative_growth_in_shock_window"

    return negative_years[0], available_shock_values, "negative_growth_found"


def compute_recovery_for_country(country, country_name, country_df, config):
    shock_id = config["shock_id"]
    recovery_column = config["recovery_column"]
    shock_window_years = config["shock_window_years"]
    not_recovered_value = config["not_recovered_value"]

    growth_by_year = build_country_growth_map(country_df)

    first_negative_year, shock_window_values, shock_detection_status = determine_first_negative_year(
        growth_by_year,
        shock_window_years,
    )

    base_summary = {
        "Country": country,
        "country_name": country_name,
        "shock_id": shock_id,
        "recovery_column": recovery_column,
        "shock_window_years": ",".join(map(str, shock_window_years)),
        "shock_window_values": "; ".join(
            f"{year}:{growth_by_year.get(year, np.nan):.6g}"
            for year in shock_window_years
            if year in growth_by_year and pd.notna(growth_by_year.get(year))
        ),
        "shock_detection_status": shock_detection_status,
        "first_negative_growth_year": first_negative_year,
        "baseline_year": np.nan,
        "baseline_available_in_growth_file": False,
        "recovery_year": np.nan,
        "recovery_status": np.nan,
        "recovery_value": np.nan,
        "years_from_baseline_to_recovery": np.nan,
        "years_after_first_negative_to_recovery": np.nan,
        "path_stopped_due_to_missing_year": np.nan,
        "last_index_year": np.nan,
        "last_index_value": np.nan,
    }

    index_path_rows = []

    if shock_detection_status == "no_shock_window_data":
        base_summary["recovery_status"] = "excluded_no_shock_window_data"
        return base_summary, index_path_rows

    if shock_detection_status == "no_negative_growth_in_shock_window":
        base_summary["recovery_status"] = "no_negative_growth_in_shock_window"
        base_summary["recovery_value"] = 0
        base_summary["years_from_baseline_to_recovery"] = 0
        base_summary["years_after_first_negative_to_recovery"] = 0
        return base_summary, index_path_rows

    baseline_year = first_negative_year - 1
    base_summary["baseline_year"] = baseline_year
    base_summary["baseline_available_in_growth_file"] = baseline_year in growth_by_year

    # Create a real-GDP index path starting at 100 in the dynamic baseline year.
    current_index = 100.0

    index_path_rows.append({
        "Country": country,
        "country_name": country_name,
        "shock_id": shock_id,
        "Year": baseline_year,
        "gdp_growth": np.nan,
        "gdp_index_dynamic_baseline_100": current_index,
        "is_baseline_year": True,
        "first_negative_growth_year": first_negative_year,
        "baseline_year": baseline_year,
    })

    recovery_year = None
    missing_year_that_stopped_path = None

    max_observed_year = int(country_df["Year"].max())

    for year in range(baseline_year + 1, max_observed_year + 1):
        if year not in growth_by_year or pd.isna(growth_by_year.get(year)):
            missing_year_that_stopped_path = year
            break

        growth = float(growth_by_year[year])
        current_index = current_index * (1 + growth / 100.0)

        index_path_rows.append({
            "Country": country,
            "country_name": country_name,
            "shock_id": shock_id,
            "Year": year,
            "gdp_growth": growth,
            "gdp_index_dynamic_baseline_100": current_index,
            "is_baseline_year": False,
            "first_negative_growth_year": first_negative_year,
            "baseline_year": baseline_year,
        })

        if year >= first_negative_year and current_index >= 100.0:
            recovery_year = year
            break

    base_summary["path_stopped_due_to_missing_year"] = missing_year_that_stopped_path

    if index_path_rows:
        base_summary["last_index_year"] = index_path_rows[-1]["Year"]
        base_summary["last_index_value"] = index_path_rows[-1]["gdp_index_dynamic_baseline_100"]

    if recovery_year is not None:
        years_from_baseline = recovery_year - baseline_year
        years_after_first_negative = recovery_year - first_negative_year

        base_summary["recovery_year"] = recovery_year
        base_summary["recovery_status"] = "recovered"
        base_summary["years_from_baseline_to_recovery"] = years_from_baseline
        base_summary["years_after_first_negative_to_recovery"] = years_after_first_negative

        if FINAL_RECOVERY_TIME_DEFINITION == "years_after_first_negative_growth_year":
            base_summary["recovery_value"] = years_after_first_negative
        elif FINAL_RECOVERY_TIME_DEFINITION == "years_from_baseline_year":
            base_summary["recovery_value"] = years_from_baseline
        else:
            raise ValueError(f"Unknown recovery time definition: {FINAL_RECOVERY_TIME_DEFINITION}")

    else:
        if missing_year_that_stopped_path is not None:
            base_summary["recovery_status"] = "insufficient_growth_path_missing_year_before_recovery"
            base_summary["recovery_value"] = np.nan
        else:
            base_summary["recovery_status"] = "not_recovered_by_available_data"
            base_summary["recovery_value"] = not_recovered_value

    return base_summary, index_path_rows

In [6]:
# ------------------------------------------------------
# RUN RECOVERY CONSTRUCTION
# ------------------------------------------------------

summary_rows = []
index_path_rows = []

for country, country_df in gdp_growth.groupby("Country"):
    country_df = country_df.sort_values("Year").copy()
    country_name = country_df["country_name"].dropna().iloc[0] if country_df["country_name"].notna().any() else country

    for config in SHOCK_CONFIGS:
        summary, path_rows = compute_recovery_for_country(
            country=country,
            country_name=country_name,
            country_df=country_df,
            config=config,
        )

        summary_rows.append(summary)
        index_path_rows.extend(path_rows)

recovery_long = pd.DataFrame(summary_rows)
recovery_index_paths = pd.DataFrame(index_path_rows)

recovery_long.to_csv(
    PROCESSED_DIR / "country_recovery_audit_dynamic_baseline_long.csv",
    index=False,
)

recovery_long.to_csv(
    DIAGNOSTICS_DIR / "country_recovery_audit_dynamic_baseline_long.csv",
    index=False,
)

recovery_index_paths.to_csv(
    PROCESSED_DIR / "country_recovery_index_paths_dynamic_baseline.csv",
    index=False,
)

recovery_index_paths.to_csv(
    DIAGNOSTICS_DIR / "country_recovery_index_paths_dynamic_baseline.csv",
    index=False,
)

print("Recovery long rows:", len(recovery_long))
print("Index path rows:", len(recovery_index_paths))

display(recovery_long.head(80))

Recovery long rows: 88
Index path rows: 355


,Country,country_name,shock_id,recovery_column,shock_window_years,shock_window_values,shock_detection_status,first_negative_growth_year,baseline_year,baseline_available_in_growth_file,recovery_year,recovery_status,recovery_value,years_from_baseline_to_recovery,years_after_first_negative_to_recovery,path_stopped_due_to_missing_year,last_index_year,last_index_value
0,AUS,Australia,2007,Recovery_2007,"2008,2009",2008:1.95485; 2009:2.21005,no_negative_growth_in_shock_window,NaN,NaN,False,NaN,no_negative_growth_in_shock_window,0.0000,0.0000,0.0000,NaN,NaN,NaN
1,AUS,Australia,2019,Recovery_2019,"2020,2021",2020:2.00695; 2021:4.25305,no_negative_growth_in_shock_window,NaN,NaN,False,NaN,no_negative_growth_in_shock_window,0.0000,0.0000,0.0000,NaN,NaN,NaN
2,AUT,Austria,2007,Recovery_2007,"2008,2009",2008:1.45331; 2009:-3.58626,negative_growth_found,2009.0000,2008.0000,True,2011.0000,recovered,2.0000,3.0000,2.0000,NaN,2011.0000,101.0314
3,AUT,Austria,2019,Recovery_2019,"2020,2021",2020:-6.31826; 2021:4.92309,negative_growth_found,2020.0000,2019.0000,True,2022.0000,recovered,2.0000,3.0000,2.0000,NaN,2022.0000,103.5338
4,BEL,Belgium,2007,Recovery_2007,"2008,2009",2008:0.446905; 2009:-1.90644,negative_growth_found,2009.0000,2008.0000,True,2010.0000,recovered,1.0000,2.0000,1.0000,NaN,2010.0000,100.7522
5,BEL,Belgium,2019,Recovery_2019,"2020,2021",2020:-4.79298; 2021:6.25446,negative_growth_found,2020.0000,2019.0000,True,2021.0000,recovered,1.0000,2.0000,1.0000,NaN,2021.0000,101.1617
6,BRA,Brazil,2007,Recovery_2007,"2008,2009",2008:5.0942; 2009:-0.125812,negative_growth_found,2009.0000,2008.0000,True,2010.0000,recovered,1.0000,2.0000,1.0000,NaN,2010.0000,107.3929
7,BRA,Brazil,2019,Recovery_2019,"2020,2021",2020:-3.27676; 2021:4.7626,negative_growth_found,2020.0000,2019.0000,True,2021.0000,recovered,1.0000,2.0000,1.0000,NaN,2021.0000,101.3298
8,CAN,Canada,2007,Recovery_2007,"2008,2009",2008:0.995406; 2009:-2.91509,negative_growth_found,2009.0000,2008.0000,True,2010.0000,recovered,1.0000,2.0000,1.0000,NaN,2010.0000,100.0856
9,CAN,Canada,2019,Recovery_2019,"2020,2021",2020:-5.03823; 2021:5.95053,negative_growth_found,2020.0000,2019.0000,True,2021.0000,recovered,1.0000,2.0000,1.0000,NaN,2021.0000,100.6125


In [7]:
# ------------------------------------------------------
# CREATE WIDE COUNTRY RECOVERY SUMMARY
# ------------------------------------------------------

summary_base = (
    recovery_long
    .pivot_table(
        index=["Country", "country_name"],
        columns="recovery_column",
        values="recovery_value",
        aggfunc="first",
    )
    .reset_index()
)

# Add status and audit fields for each shock.
audit_fields = [
    "recovery_status",
    "shock_detection_status",
    "first_negative_growth_year",
    "baseline_year",
    "baseline_available_in_growth_file",
    "recovery_year",
    "years_from_baseline_to_recovery",
    "years_after_first_negative_to_recovery",
    "path_stopped_due_to_missing_year",
    "last_index_year",
    "last_index_value",
]

wide = summary_base.copy()

for config in SHOCK_CONFIGS:
    recovery_col = config["recovery_column"]
    shock_subset = recovery_long[recovery_long["recovery_column"] == recovery_col].copy()

    for field in audit_fields:
        temp = (
            shock_subset
            .pivot_table(
                index=["Country", "country_name"],
                values=field,
                aggfunc="first",
            )
            .reset_index()
            .rename(columns={field: f"{recovery_col}_{field}"})
        )

        wide = wide.merge(temp, on=["Country", "country_name"], how="left")

# Ensure expected recovery columns exist even if pivot omitted them.
for config in SHOCK_CONFIGS:
    recovery_col = config["recovery_column"]
    if recovery_col not in wide.columns:
        wide[recovery_col] = np.nan

# Put main recovery columns near the front.
front_cols = ["Country", "country_name"] + [config["recovery_column"] for config in SHOCK_CONFIGS]
other_cols = [col for col in wide.columns if col not in front_cols]

country_recovery_summary = wide[front_cols + other_cols].copy()

country_recovery_summary = country_recovery_summary.sort_values("Country").reset_index(drop=True)

country_recovery_summary.to_csv(
    PROCESSED_DIR / "country_recovery_summary_dynamic_baseline_2007_2019.csv",
    index=False,
)

country_recovery_summary.to_csv(
    DIAGNOSTICS_DIR / "country_recovery_summary_dynamic_baseline_2007_2019.csv",
    index=False,
)

display(country_recovery_summary.head(80))

,Country,country_name,Recovery_2007,Recovery_2019,Recovery_2007_recovery_status,Recovery_2007_shock_detection_status,Recovery_2007_first_negative_growth_year,Recovery_2007_baseline_year,Recovery_2007_baseline_available_in_growth_file,Recovery_2007_recovery_year,Recovery_2007_years_from_baseline_to_recovery,Recovery_2007_years_after_first_negative_to_recovery,Recovery_2007_last_index_year,Recovery_2007_last_index_value,Recovery_2019_recovery_status,Recovery_2019_shock_detection_status,Recovery_2019_first_negative_growth_year,Recovery_2019_baseline_year,Recovery_2019_baseline_available_in_growth_file,Recovery_2019_recovery_year,Recovery_2019_years_from_baseline_to_recovery,Recovery_2019_years_after_first_negative_to_recovery,Recovery_2019_last_index_year,Recovery_2019_last_index_value
0,AUS,Australia,0.0000,0.0000,no_negative_growth_in_shock_window,no_negative_growth_in_shock_window,NaN,NaN,False,NaN,0.0000,0.0000,NaN,NaN,no_negative_growth_in_shock_window,no_negative_growth_in_shock_window,NaN,NaN,False,NaN,0.0000,0.0000,NaN,NaN
1,AUT,Austria,2.0000,2.0000,recovered,negative_growth_found,2009.0000,2008.0000,True,2011.0000,3.0000,2.0000,2011.0000,101.0314,recovered,negative_growth_found,2020.0000,2019.0000,True,2022.0000,3.0000,2.0000,2022.0000,103.5338
2,BEL,Belgium,1.0000,1.0000,recovered,negative_growth_found,2009.0000,2008.0000,True,2010.0000,2.0000,1.0000,2010.0000,100.7522,recovered,negative_growth_found,2020.0000,2019.0000,True,2021.0000,2.0000,1.0000,2021.0000,101.1617
3,BRA,Brazil,1.0000,1.0000,recovered,negative_growth_found,2009.0000,2008.0000,True,2010.0000,2.0000,1.0000,2010.0000,107.3929,recovered,negative_growth_found,2020.0000,2019.0000,True,2021.0000,2.0000,1.0000,2021.0000,101.3298
4,CAN,Canada,1.0000,1.0000,recovered,negative_growth_found,2009.0000,2008.0000,True,2010.0000,2.0000,1.0000,2010.0000,100.0856,recovered,negative_growth_found,2020.0000,2019.0000,True,2021.0000,2.0000,1.0000,2021.0000,100.6125
5,CHE,Switzerland,1.0000,1.0000,recovered,negative_growth_found,2009.0000,2008.0000,True,2010.0000,2.0000,1.0000,2010.0000,101.2626,recovered,negative_growth_found,2020.0000,2019.0000,True,2021.0000,2.0000,1.0000,2021.0000,103.7817
6,CHL,Chile,1.0000,1.0000,recovered,negative_growth_found,2009.0000,2008.0000,True,2010.0000,2.0000,1.0000,2010.0000,104.6682,recovered,negative_growth_found,2020.0000,2019.0000,True,2021.0000,2.0000,1.0000,2021.0000,104.5016
7,CHN,China,0.0000,0.0000,no_negative_growth_in_shock_window,no_negative_growth_in_shock_window,NaN,NaN,False,NaN,0.0000,0.0000,NaN,NaN,no_negative_growth_in_shock_window,no_negative_growth_in_shock_window,NaN,NaN,False,NaN,0.0000,0.0000,NaN,NaN
8,COL,Colombia,0.0000,1.0000,no_negative_growth_in_shock_window,no_negative_growth_in_shock_window,NaN,NaN,False,NaN,0.0000,0.0000,NaN,NaN,recovered,negative_growth_found,2020.0000,2019.0000,True,2021.0000,2.0000,1.0000,2021.0000,102.8391
9,CRI,Costa Rica,1.0000,1.0000,recovered,negative_growth_found,2009.0000,2008.0000,True,2010.0000,2.0000,1.0000,2010.0000,104.4401,recovered,negative_growth_found,2020.0000,2019.0000,True,2021.0000,2.0000,1.0000,2021.0000,103.3233


In [8]:
# ------------------------------------------------------
# STATUS SUMMARIES
# ------------------------------------------------------

country_recovery_status_summary = (
    recovery_long
    .groupby(["shock_id", "recovery_column", "recovery_status"])
    .agg(
        countries=("Country", "nunique"),
        min_recovery_value=("recovery_value", "min"),
        median_recovery_value=("recovery_value", "median"),
        max_recovery_value=("recovery_value", "max"),
    )
    .reset_index()
    .sort_values(["shock_id", "countries"], ascending=[True, False])
)

country_recovery_status_summary.to_csv(
    PROCESSED_DIR / "country_recovery_status_summary_dynamic_baseline.csv",
    index=False,
)

country_recovery_status_summary.to_csv(
    DIAGNOSTICS_DIR / "country_recovery_status_summary_dynamic_baseline.csv",
    index=False,
)

shock_detection_summary = (
    recovery_long
    .groupby(["shock_id", "shock_detection_status"])
    .agg(countries=("Country", "nunique"))
    .reset_index()
    .sort_values(["shock_id", "countries"], ascending=[True, False])
)

shock_detection_summary.to_csv(
    DIAGNOSTICS_DIR / "shock_detection_status_summary.csv",
    index=False,
)

display(country_recovery_status_summary)
display(shock_detection_summary)

,shock_id,recovery_column,recovery_status,countries,min_recovery_value,median_recovery_value,max_recovery_value
2,2007,Recovery_2007,recovered,35,1.0000,2.0000,15.0000
0,2007,Recovery_2007,no_negative_growth_in_shock_window,8,0.0000,0.0000,0.0000
1,2007,Recovery_2007,not_recovered_by_available_data,1,20.0000,20.0000,20.0000
5,2019,Recovery_2019,recovered,37,1.0000,1.0000,2.0000
4,2019,Recovery_2019,no_negative_growth_in_shock_window,6,0.0000,0.0000,0.0000
3,2019,Recovery_2019,excluded_no_shock_window_data,1,NaN,NaN,NaN


,shock_id,shock_detection_status,countries
0,2007,negative_growth_found,36
1,2007,no_negative_growth_in_shock_window,8
2,2019,negative_growth_found,37
3,2019,no_negative_growth_in_shock_window,6
4,2019,no_shock_window_data,1


In [9]:
# ------------------------------------------------------
# EXCLUDED / PROBLEM CASES
# ------------------------------------------------------

problem_statuses = {
    "excluded_no_shock_window_data",
    "insufficient_growth_path_missing_year_before_recovery",
}

recovery_problem_cases = recovery_long[
    recovery_long["recovery_status"].isin(problem_statuses)
].copy()

recovery_problem_cases.to_csv(
    DIAGNOSTICS_DIR / "recovery_problem_cases.csv",
    index=False,
)

not_recovered_cases = recovery_long[
    recovery_long["recovery_status"] == "not_recovered_by_available_data"
].copy()

not_recovered_cases.to_csv(
    DIAGNOSTICS_DIR / "not_recovered_by_available_data_cases.csv",
    index=False,
)

display(recovery_problem_cases)
display(not_recovered_cases.head(50))

,Country,country_name,shock_id,recovery_column,shock_window_years,shock_window_values,shock_detection_status,first_negative_growth_year,baseline_year,baseline_available_in_growth_file,recovery_year,recovery_status,recovery_value,years_from_baseline_to_recovery,years_after_first_negative_to_recovery,path_stopped_due_to_missing_year,last_index_year,last_index_value
75,RUS,Russian Federation,2019,Recovery_2019,"2020,2021",,no_shock_window_data,NaN,NaN,False,NaN,excluded_no_shock_window_data,NaN,NaN,NaN,NaN,NaN,NaN


,Country,country_name,shock_id,recovery_column,shock_window_years,shock_window_values,shock_detection_status,first_negative_growth_year,baseline_year,baseline_available_in_growth_file,recovery_year,recovery_status,recovery_value,years_from_baseline_to_recovery,years_after_first_negative_to_recovery,path_stopped_due_to_missing_year,last_index_year,last_index_value
36,GRC,Greece,2007,Recovery_2007,"2008,2009",2008:0.0574739; 2009:-4.11928,negative_growth_found,2009.0000,2008.0000,True,NaN,not_recovered_by_available_data,20.0000,NaN,NaN,NaN,2025.0000,86.1397


In [10]:
# ------------------------------------------------------
# ARGENTINA MANUAL VALIDATION SNAPSHOT
# ------------------------------------------------------
# ARG was mentioned as a useful manual validation case.

argentina_growth = gdp_growth[gdp_growth["Country"] == "ARG"].copy()
argentina_recovery_audit = recovery_long[recovery_long["Country"] == "ARG"].copy()
argentina_index_path = recovery_index_paths[recovery_index_paths["Country"] == "ARG"].copy()

argentina_growth.to_csv(
    DIAGNOSTICS_DIR / "argentina_gdp_growth_input.csv",
    index=False,
)

argentina_recovery_audit.to_csv(
    DIAGNOSTICS_DIR / "argentina_recovery_audit.csv",
    index=False,
)

argentina_index_path.to_csv(
    DIAGNOSTICS_DIR / "argentina_recovery_index_path.csv",
    index=False,
)

print("Argentina GDP growth input:")
display(argentina_growth)

print("Argentina recovery audit:")
display(argentina_recovery_audit)

print("Argentina index path:")
display(argentina_index_path)

Argentina GDP growth input:


,Country,country_name,Year,Value,source_file,source_family,concept,expected_unit


Argentina recovery audit:


,Country,country_name,shock_id,recovery_column,shock_window_years,shock_window_values,shock_detection_status,first_negative_growth_year,baseline_year,baseline_available_in_growth_file,recovery_year,recovery_status,recovery_value,years_from_baseline_to_recovery,years_after_first_negative_to_recovery,path_stopped_due_to_missing_year,last_index_year,last_index_value


Argentina index path:


,Country,country_name,shock_id,Year,gdp_growth,gdp_index_dynamic_baseline_100,is_baseline_year,first_negative_growth_year,baseline_year


In [11]:
# ------------------------------------------------------
# OUTPUT DATA DICTIONARY
# ------------------------------------------------------

data_dictionary = pd.DataFrame([
    {
        "file": "country_recovery_summary_dynamic_baseline_2007_2019.csv",
        "column": "Recovery_2007",
        "description": "Final recovery value for the 2007 shock window. 0 = no negative growth in shock window; 20 = not recovered by available data; otherwise years after first negative GDP growth year until the GDP index returns to dynamic baseline 100.",
    },
    {
        "file": "country_recovery_summary_dynamic_baseline_2007_2019.csv",
        "column": "Recovery_2019",
        "description": "Final recovery value for the 2019 shock window. 0 = no negative growth in shock window; 20 = not recovered by available data; otherwise years after first negative GDP growth year until the GDP index returns to dynamic baseline 100.",
    },
    {
        "file": "country_recovery_audit_dynamic_baseline_long.csv",
        "column": "first_negative_growth_year",
        "description": "First year with negative real GDP growth inside the configured shock window.",
    },
    {
        "file": "country_recovery_audit_dynamic_baseline_long.csv",
        "column": "baseline_year",
        "description": "Dynamic pre-shock baseline year, defined as first_negative_growth_year - 1.",
    },
    {
        "file": "country_recovery_index_paths_dynamic_baseline.csv",
        "column": "gdp_index_dynamic_baseline_100",
        "description": "Reconstructed GDP index from annual growth rates, set to 100 in the dynamic baseline year.",
    },
    {
        "file": "country_recovery_audit_dynamic_baseline_long.csv",
        "column": "years_from_baseline_to_recovery",
        "description": "Recovery year minus baseline year. Exported for transparency; not the default final Recovery_* value.",
    },
    {
        "file": "country_recovery_audit_dynamic_baseline_long.csv",
        "column": "years_after_first_negative_to_recovery",
        "description": "Recovery year minus first negative growth year. This is the default final Recovery_* value when a country recovers.",
    },
])

data_dictionary.to_csv(
    PROCESSED_DIR / "country_recovery_data_dictionary.csv",
    index=False,
)

data_dictionary.to_csv(
    DIAGNOSTICS_DIR / "country_recovery_data_dictionary.csv",
    index=False,
)

display(data_dictionary)

,file,column,description
0,country_recovery_summary_dynamic_baseline_2007...,Recovery_2007,Final recovery value for the 2007 shock window...
1,country_recovery_summary_dynamic_baseline_2007...,Recovery_2019,Final recovery value for the 2019 shock window...
2,country_recovery_audit_dynamic_baseline_long.csv,first_negative_growth_year,First year with negative real GDP growth insid...
3,country_recovery_audit_dynamic_baseline_long.csv,baseline_year,"Dynamic pre-shock baseline year, defined as fi..."
4,country_recovery_index_paths_dynamic_baseline.csv,gdp_index_dynamic_baseline_100,Reconstructed GDP index from annual growth rat...
5,country_recovery_audit_dynamic_baseline_long.csv,years_from_baseline_to_recovery,Recovery year minus baseline year. Exported fo...
6,country_recovery_audit_dynamic_baseline_long.csv,years_after_first_negative_to_recovery,Recovery year minus first negative growth year...


In [12]:
# ------------------------------------------------------
# CREATE RECOVERY AUDIT WORKBOOK
# ------------------------------------------------------

RECOVERY_AUDIT_XLSX = AUDIT_DIR / "02_GDP_Recovery_Dynamic_Baseline_Audit.xlsx"

audit_sources = [
    {
        "sheet_name": "01_recovery_summary",
        "description": "Final country-level wide recovery summary.",
        "path": PROCESSED_DIR / "country_recovery_summary_dynamic_baseline_2007_2019.csv",
    },
    {
        "sheet_name": "02_recovery_audit",
        "description": "Long-format recovery audit by country and shock.",
        "path": PROCESSED_DIR / "country_recovery_audit_dynamic_baseline_long.csv",
    },
    {
        "sheet_name": "03_status_summary",
        "description": "Recovery status summary by shock.",
        "path": PROCESSED_DIR / "country_recovery_status_summary_dynamic_baseline.csv",
    },
    {
        "sheet_name": "04_shock_detection",
        "description": "Shock detection status summary.",
        "path": DIAGNOSTICS_DIR / "shock_detection_status_summary.csv",
    },
    {
        "sheet_name": "05_problem_cases",
        "description": "Cases with no shock-window data or insufficient growth paths.",
        "path": DIAGNOSTICS_DIR / "recovery_problem_cases.csv",
    },
    {
        "sheet_name": "06_not_recovered",
        "description": "Countries not recovered by available data.",
        "path": DIAGNOSTICS_DIR / "not_recovered_by_available_data_cases.csv",
    },
    {
        "sheet_name": "07_gdp_coverage",
        "description": "GDP growth input coverage by country.",
        "path": DIAGNOSTICS_DIR / "gdp_growth_country_coverage_summary.csv",
    },
    {
        "sheet_name": "08_duplicates",
        "description": "GDP growth duplicate Country-Year check.",
        "path": DIAGNOSTICS_DIR / "gdp_growth_duplicate_country_year_check.csv",
    },
    {
        "sheet_name": "09_argentina_audit",
        "description": "Argentina manual validation recovery audit.",
        "path": DIAGNOSTICS_DIR / "argentina_recovery_audit.csv",
    },
    {
        "sheet_name": "10_data_dictionary",
        "description": "Recovery output data dictionary.",
        "path": PROCESSED_DIR / "country_recovery_data_dictionary.csv",
    },
]


def safe_sheet_name(name, used):
    cleaned = re.sub(r"[/\\*\[\]:?]", "_", str(name))[:31]
    base = cleaned
    i = 1

    while cleaned in used:
        suffix = f"_{i}"
        cleaned = base[:31 - len(suffix)] + suffix
        i += 1

    used.add(cleaned)
    return cleaned


def estimate_width(series, col_name, min_width=10, max_width=60):
    values = [str(col_name)] + series.dropna().astype(str).head(200).tolist()

    if not values:
        return min_width

    return max(min_width, min(max(len(v) for v in values) + 2, max_width))


used_sheets = set()

with pd.ExcelWriter(RECOVERY_AUDIT_XLSX, engine="xlsxwriter") as writer:
    workbook = writer.book

    title_fmt = workbook.add_format({
        "bold": True,
        "font_size": 15,
        "font_color": "white",
        "bg_color": "#1F4E78",
        "align": "left",
        "valign": "vcenter",
    })

    desc_fmt = workbook.add_format({
        "text_wrap": True,
        "font_size": 10,
        "font_color": "#333333",
        "valign": "top",
    })

    header_fmt = workbook.add_format({
        "bold": True,
        "font_color": "white",
        "bg_color": "#5B9BD5",
        "border": 1,
        "align": "center",
        "valign": "vcenter",
        "text_wrap": True,
    })

    index_rows = []

    for item in audit_sources:
        path = Path(item["path"])

        if path.exists():
            try:
                df_temp = pd.read_csv(path)
                rows = len(df_temp)
                cols = len(df_temp.columns)
            except Exception:
                rows = np.nan
                cols = np.nan
        else:
            rows = 0
            cols = 0

        index_rows.append({
            "Sheet": item["sheet_name"],
            "Rows": rows,
            "Columns": cols,
            "Description": item["description"],
            "Path": str(path),
        })

    index_df = pd.DataFrame(index_rows)
    index_df.to_excel(writer, sheet_name="00_INDEX", index=False, startrow=5)

    ws = writer.sheets["00_INDEX"]
    ws.merge_range(0, 0, 0, max(4, len(index_df.columns) - 1), "02 GDP Recovery Dynamic Baseline Audit", title_fmt)
    ws.merge_range(1, 0, 3, max(4, len(index_df.columns) - 1), "Audit workbook for GDP recovery outcome construction.", desc_fmt)

    for col_idx, col_name in enumerate(index_df.columns):
        ws.write(5, col_idx, col_name, header_fmt)
        ws.set_column(col_idx, col_idx, estimate_width(index_df[col_name], col_name))

    ws.autofilter(5, 0, 5 + len(index_df), len(index_df.columns) - 1)
    ws.freeze_panes(6, 0)

    for item in audit_sources:
        path = Path(item["path"])

        if path.exists():
            try:
                df_sheet = pd.read_csv(path)
            except Exception as e:
                df_sheet = pd.DataFrame({"message": [f"Could not read file: {e}"]})
        else:
            df_sheet = pd.DataFrame({"message": ["File not found."]})

        if df_sheet.empty:
            df_sheet = pd.DataFrame({"message": ["No rows in this table."]})

        sheet_name = safe_sheet_name(item["sheet_name"], used_sheets)

        df_sheet.to_excel(writer, sheet_name=sheet_name, index=False, startrow=4)

        ws = writer.sheets[sheet_name]
        max_col = max(4, len(df_sheet.columns) - 1)

        ws.merge_range(0, 0, 0, max_col, item["sheet_name"], title_fmt)
        ws.merge_range(1, 0, 3, max_col, item["description"], desc_fmt)

        for col_idx, col_name in enumerate(df_sheet.columns):
            ws.write(4, col_idx, col_name, header_fmt)
            ws.set_column(col_idx, col_idx, estimate_width(df_sheet[col_name], col_name))

        ws.autofilter(4, 0, 4 + len(df_sheet), len(df_sheet.columns) - 1)
        ws.freeze_panes(5, 0)

print("Recovery audit workbook created:")
print(RECOVERY_AUDIT_XLSX.resolve())

Recovery audit workbook created:
D:\Milano Bicocca\Course Materials\1st Year\02 - Data Science Lab - 2526-1-FDS02Q003\03 Projects\Index\Notebooks\Data\Audit\02_GDP_Recovery_Dynamic_Baseline_Audit.xlsx


In [13]:
# ------------------------------------------------------
# FINAL COMPLETION CHECK
# ------------------------------------------------------

print("02 GDP RECOVERY DYNAMIC BASELINE COMPLETE")
print("=" * 80)

print("\nInput file:")
print(INPUT_FILE.resolve())

print("\nProcessed folder:")
print(PROCESSED_DIR.resolve())

print("\nDiagnostics folder:")
print(DIAGNOSTICS_DIR.resolve())

print("\nAudit workbook:")
print(RECOVERY_AUDIT_XLSX.resolve())

print("\nMain outputs:")
main_outputs = [
    "country_recovery_summary_dynamic_baseline_2007_2019.csv",
    "country_recovery_audit_dynamic_baseline_long.csv",
    "country_recovery_index_paths_dynamic_baseline.csv",
    "country_recovery_status_summary_dynamic_baseline.csv",
    "country_recovery_data_dictionary.csv",
]

for file_name in main_outputs:
    path = PROCESSED_DIR / file_name
    status = "FOUND" if path.exists() else "MISSING"
    print(f"- {status}: {file_name}")

print("\nRecovery status summary:")
display(country_recovery_status_summary)

print("\nImportant notes:")
print("1. Recovery outcomes use GDP growth only.")
print("2. Dynamic baseline is selected from the first negative growth year in each shock window.")
print("3. No negative growth in the shock window is encoded as 0.")
print("4. Not recovered by available data is encoded as 20.")
print("5. The default recovered-country value is years after first negative GDP growth year.")
print("6. Recovery outputs are for validation/interpretation later, not POSet ordering.")

print("\nNext notebook:")
print("03_WGI_Governance_Compilation.ipynb")

02 GDP RECOVERY DYNAMIC BASELINE COMPLETE

Input file:
D:\Milano Bicocca\Course Materials\1st Year\02 - Data Science Lab - 2526-1-FDS02Q003\03 Projects\Index\Notebooks\Data\Processed\00_Comparable_Raw_Files\Combined_Long\all_comparable_sources_long.csv

Processed folder:
D:\Milano Bicocca\Course Materials\1st Year\02 - Data Science Lab - 2526-1-FDS02Q003\03 Projects\Index\Notebooks\Data\Processed\Recovery

Diagnostics folder:
D:\Milano Bicocca\Course Materials\1st Year\02 - Data Science Lab - 2526-1-FDS02Q003\03 Projects\Index\Notebooks\Data\Diagnostics\02_GDP_Recovery_Dynamic_Baseline

Audit workbook:
D:\Milano Bicocca\Course Materials\1st Year\02 - Data Science Lab - 2526-1-FDS02Q003\03 Projects\Index\Notebooks\Data\Audit\02_GDP_Recovery_Dynamic_Baseline_Audit.xlsx

Main outputs:
- FOUND: country_recovery_summary_dynamic_baseline_2007_2019.csv
- FOUND: country_recovery_audit_dynamic_baseline_long.csv
- FOUND: country_recovery_index_paths_dynamic_baseline.csv
- FOUND: country_recovery

,shock_id,recovery_column,recovery_status,countries,min_recovery_value,median_recovery_value,max_recovery_value
2,2007,Recovery_2007,recovered,35,1.0000,2.0000,15.0000
0,2007,Recovery_2007,no_negative_growth_in_shock_window,8,0.0000,0.0000,0.0000
1,2007,Recovery_2007,not_recovered_by_available_data,1,20.0000,20.0000,20.0000
5,2019,Recovery_2019,recovered,37,1.0000,1.0000,2.0000
4,2019,Recovery_2019,no_negative_growth_in_shock_window,6,0.0000,0.0000,0.0000
3,2019,Recovery_2019,excluded_no_shock_window_data,1,NaN,NaN,NaN



Important notes:
1. Recovery outcomes use GDP growth only.
2. Dynamic baseline is selected from the first negative growth year in each shock window.
3. No negative growth in the shock window is encoded as 0.
4. Not recovered by available data is encoded as 20.
5. The default recovered-country value is years after first negative GDP growth year.
6. Recovery outputs are for validation/interpretation later, not POSet ordering.

Next notebook:
03_WGI_Governance_Compilation.ipynb
